# All image generate

In [1]:
import cv2
import numpy as np

def overlay_heatmap(image, saliency, alpha=0.5):

    # image = (image - image.min()) / (image.max() - image.min() + 1e-8)
    # saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    image_uint8 = np.uint8(255 * image)
    heatmap_uint8 = np.uint8(255 * saliency)

    heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(
        heatmap,
        alpha,
        np.stack([image_uint8]*3, axis=-1),
        1-alpha,
        0
    )

    return overlay

In [10]:
# Delete Files with "Slice Weighted Rollout" in the name
# import os

# root_dir = "/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image"

# deleted_count = 0

# for dirpath, dirnames, filenames in os.walk(root_dir):
#     for filename in filenames:
#         if "Slice Weighted Rollout" in filename:
#             file_path = os.path.join(dirpath, filename)
            
#             try:
#                 os.remove(file_path)
#                 print(f"Deleted: {file_path}")
#                 deleted_count += 1
#             except Exception as e:
#                 print(f"Error deleting {file_path}: {e}")

# print(f"\nTotal deleted files: {deleted_count}")

In [3]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import gc
from pathlib import Path
import sys

PROJECT_ROOT = Path("/home/jovyan/work/MST")
sys.path.append(str(PROJECT_ROOT))

from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D

ds = ODELIA_Dataset3D(
    path_root="/home/jovyan/work/MST/mst/data/datasets/ODELIA_datasets",
    split="test"
)


In [4]:
import json

## Load the JSON file containing the correct UIDs for each class
with open("/home/jovyan/work/MST/scripts/Jupyter_notebook/Confusion Matrix/v3_correct_full.json", "r") as f:
    correct_data = json.load(f)

with open("/home/jovyan/work/MST/scripts/Jupyter_notebook/Confusion Matrix/v3_incorrect_full.json", "r") as f:
    incorrect_data = json.load(f)

saliency_set = {"GradCAM": "gradcam", "Raw Attention": "last_layer", "Slice Weighted Rollout": "slice_weighted_rollout"}
# saliency_set = {"Slice Weighted Rollout": "slice_weighted_rollout"}



In [13]:
# data = {'0': ['ODELIA_TRICKS_0091_1_right']}

In [14]:
# def build_uid_index(dataset):
#     return {dataset[i]["uid"]: i for i in range(len(dataset))}

# uid_to_index = build_uid_index(ds)

In [ ]:

# with open("uid_to_index.pkl", "wb") as f:
#     pickle.dump(uid_to_index, f)

In [8]:
import pickle
with open("uid_to_index.pkl", "rb") as f:
    uid_to_index = pickle.load(f)

In [9]:


def files_exist(class_save_root, pred_class_name, file_name, formats):
    base_path = os.path.join(class_save_root, str(pred_class_name))
    for fmt in formats:
        fmt_folder = os.path.join(base_path, fmt[1:])
        save_path = os.path.join(fmt_folder, file_name + fmt)
        if not os.path.exists(save_path):
            return False
    return True

def save_saliency_visualizations(
    data,
    saliency_set,
    dataset,
    overlay_heatmap,
    save_root,
):

    os.makedirs(save_root, exist_ok=True)

    for gt_class, samples in data.items():
        class_save_root = os.path.join(save_root, gt_class)
        os.makedirs(class_save_root, exist_ok=True)


        for sample in samples:
            uid = sample["UID"]
            gt_class_name = sample["GT"]
            pred_class_name = sample["NN"]

            input = None
            idx = uid_to_index.get(uid)

            if idx is None:
                print(f"[ERROR] UID not found in dataset: {uid}")
                continue
            
            input = dataset[idx]
            num_slices = input["source"].shape[1]  # e.g., 32 slices

            volume = input["source"].squeeze(0).cpu()  # from [1,32,224,224] to [32,224,224]
            volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
            label = input["target"] # ground truth class index
            
            
            for method_name, saliency_map in saliency_set.items():

                file_formats = [".png", ".jpg"]
            
                overlay_file = f"{uid}_pred_{pred_class_name}_GT_{label}_{method_name}_overlay"
                comparison_file = f"{uid}_pred_{pred_class_name}_GT_{label}_{method_name}_comparison"
            
                # Skip early if BOTH outputs already exist
                overlay_done = files_exist(class_save_root, pred_class_name, overlay_file, file_formats)
                comparison_done = files_exist(class_save_root, pred_class_name, comparison_file, file_formats)
            
                if overlay_done and comparison_done:
                    print(f"[SKIP] Already processed: {uid} | {method_name}")
                    continue

                # -----------------------------
                # LOAD SALIENCY
                # -----------------------------
                saliency_path = (
                    f"/home/jovyan/work/MST/results/DINOv3ViTB/saliency_results/"
                    f"{saliency_map}/class_{label}/pt/{uid}_importance.pt"
                )

                if not os.path.exists(saliency_path):
                    print(f"[WARNING] Missing saliency: {saliency_path}")
                    continue

                saliency = torch.load(saliency_path, map_location="cpu", weights_only=True).numpy()

                print(f"Processing: {uid} | GT {label} | Pred {pred_class_name} | {method_name}")

                # =========================================================
                # 1) OVERLAY GRID
                # =========================================================
                overlays = []
                for i in range(num_slices):
                    overlays.append(
                        overlay_heatmap(volume[i], saliency[i])
                    )

                fig, axes = plt.subplots(4, 8, figsize=(20, 10))
                for i, ax in enumerate(axes.flat):
                    ax.imshow(overlays[i])
                    ax.set_title(f"S{i+1}", fontsize=8)
                    ax.axis("off")

                #fig.suptitle(f"{uid} | GT {label} | Pred {pred_class_name} | {method_name}", fontsize=14)


                plt.tight_layout()
                for fmt in file_formats:
                    fmt_folder = os.path.join(class_save_root, str(pred_class_name), fmt[1:])  # "png", "jpg"
                    os.makedirs(fmt_folder, exist_ok=True)

                    save_path = os.path.join(
                        fmt_folder,
                        overlay_file + fmt
                    )

                    plt.savefig(save_path, dpi=150 if fmt == ".png" else 100)
                # plt.savefig(overlay_path + ".png", dpi=150)
                # plt.savefig(overlay_path + ".jpg", dpi= 100)
                plt.close(fig)

                # =========================================================
                # 2) COMPARISON (RAW + OVERLAY)
                # =========================================================
                comparisons = []
                for i in range(num_slices):

                    img = volume[i]
                    sal = saliency[i]

                    overlay = overlay_heatmap(img, sal)

                    img_rgb = np.stack([img]*3, axis=-1)
                    # img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() + 1e-8)
                    img_rgb = np.uint8(255 * img_rgb)

                    combined = np.concatenate([img_rgb, overlay], axis=1)
                    comparisons.append(combined)

                fig, axes = plt.subplots(8, 4, figsize=(10, 12))
                for i, ax in enumerate(axes.flat):
                    ax.imshow(comparisons[i])
                    ax.set_title(f"S{i+1}", fontsize=8)
                    ax.axis("off")

                # fig.suptitle(f"{uid} | GT {label} | Pred {pred_class_name} | {method_name}", fontsize=14)


                plt.tight_layout()
                for fmt in file_formats:
                    fmt_folder = os.path.join(class_save_root, str(pred_class_name), fmt[1:])  # "png", "jpg"
                    os.makedirs(fmt_folder, exist_ok=True)

                    save_path = os.path.join(
                        fmt_folder,
                        comparison_file + fmt
                    )

                    plt.savefig(save_path, dpi=150 if fmt == ".png" else 100)
                # plt.savefig(comparison_path/"" + ".png", dpi=150)
                # plt.savefig(comparison_path + ".jpg", dpi = 100)
                plt.close(fig)

                # -----------------------------
                # MEMORY CLEANUP
                # -----------------------------
                del overlays, comparisons, saliency
                torch.cuda.empty_cache()
                gc.collect()
    print ("------------------------------------------------")
    print ("Finished processing all samples.")
    print ("------------------------------------------------")

In [18]:
save_saliency_visualizations(
    data=correct_data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image"
)

Processing: ODELIA_BRAID1_0246_1_left | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_TRICKS_0067_1_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0187_1_left | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0777_1_left | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_TRICKS_0101_1_left | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0187_1_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0242_1_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0243_1_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_TRICKS_0080_1_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: ODELIA_TRICKS_0112_1_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: EF11BBDA_left | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: 9C5093A5_right | GT 0 | Pred 0 | Slice Weighted Rollout
Processing: 02F4A1FB_left | GT 0 | Pred 0 | Slice Weighted Rollout
Pr

In [19]:
save_saliency_visualizations(
    data=incorrect_data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image"
)

Processing: ODELIA_TRICKS_0091_1_left | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: 08FEB48B_left | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: 3E6C31E4_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: RUMC_069_left | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: RUMC_035_right | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: UKA_11_left | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_58_left | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_64_left | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_71_left | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_12_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_3_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_48_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_55_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: UKA_78_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: ODELIA_BRAID1_0246_1_right | GT 1 | Pred 0 | Slice

# Select Specific pictures

In [13]:
def save_selected_slice_comparison(
    data,
    saliency_set,
    dataset,
    overlay_heatmap,
    save_root,
    slice_indices
):

    os.makedirs(save_root, exist_ok=True)

    for topic, samples in data.items():
        class_save_root = os.path.join(save_root, topic)
        os.makedirs(class_save_root, exist_ok=True)

        for uid in samples:

            idx = uid_to_index.get(uid)
            if idx is None:
                print(f"[ERROR] UID not found: {uid}")
                continue

            input = dataset[idx]

            volume = input["source"].squeeze(0).cpu().numpy()
            volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

            label = input["target"]

            # =========================
            # LOAD ALL SALIENCY MAPS
            # =========================
            saliencys = {}

            for method_name, saliency_map in saliency_set.items():

                saliency_path = (
                    f"/home/jovyan/work/MST/results/DINOv3ViTB/saliency_results/"
                    f"{saliency_map}/class_{label}/pt/{uid}_importance.pt"
                )

                if not os.path.exists(saliency_path):
                    print(f"[WARNING] Missing: {saliency_path}")
                    continue

                sal = torch.load(saliency_path, map_location="cpu", weights_only=True).numpy()

                saliencys[method_name] = sal


            # =========================
            # PLOT COMPARISON
            # =========================
            methods = list(saliencys.keys())
            num_rows = len(slice_indices)
            num_cols = len(methods) + 1  # +1 input

            fig, axes = plt.subplots(
                num_rows,
                num_cols,
                figsize=(3*num_cols, 3*num_rows),
                gridspec_kw={"wspace": 0.1, "hspace": 0}
            )

            # Ensure 2D
            if num_rows == 1:
                axes = np.expand_dims(axes, axis=0)

            # =========================
            # COLUMN HEADERS (top row)
            # =========================
            axes[0, 0].set_title("Input image", fontsize=13, pad=5)

            for c, method in enumerate(methods):
                axes[0, c+1].set_title(method, fontsize=13, pad=5)

            # =========================
            # PLOT IMAGES
            # =========================
            for r, s in enumerate(slice_indices):

                img = volume[s]
                img_rgb = np.stack([img]*3, axis=-1)
                img_rgb = np.uint8(255 * img_rgb)

                # --- INPUT COLUMN ---
                axes[r, 0].imshow(img_rgb)
                axes[r, 0].axis("off")

                # --- ROW LABEL (slice number) ---
                axes[r, 0].text(
                    -0.1, 0.5, f"Slice {s+1}",
                    transform=axes[r, 0].transAxes,
                    fontsize=13,
                    va="center",
                    ha="right"
                )

                # --- METHODS ---
                for c, method in enumerate(methods):

                    sal = saliencys[method][s]

                    # normalize (important)
                    # sal = (sal - sal.min()) / (sal.max() + 1e-8)

                    overlay = overlay_heatmap(img, sal)

                    axes[r, c+1].imshow(overlay)
                    axes[r, c+1].axis("off")

            # remove margins completely
            plt.subplots_adjust(left=0, right=1, top=0.95, bottom=0)

            save_path = os.path.join(class_save_root, f"{uid}_selected_slices.jpg")
            plt.savefig(save_path, dpi=100, bbox_inches="tight", pad_inches=0)
            plt.close(fig)

            # cleanup
            del saliencys
            torch.cuda.empty_cache()
            gc.collect()

    print("Finished.")

## Malignant case

In [11]:
saliency_set = {
    "Grad-CAM": "gradcam",
    "Last-layer attention": "last_layer",
    "Attention rollout": "slice_weighted_rollout"
}


In [21]:
## Malignant case
## 2-2, 2-0, 0-0, 1-0
data = {
    '2-2': ['ODELIA_TRICKS_0067_1_left'],
    # '2-0': ['3192387442_right'],
    # '0-0': ['ODELIA_BRAID1_0246_1_left'],
    # '1-0': ['9C5093A5_left']
}

slice_indices = [7, 10, 14, 16]  # choose manually
save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [22]:
data = {'2-0': ['3192387442_right'],
    # '0-0': ['ODELIA_BRAID1_0246_1_left'],
    # '1-0': ['9C5093A5_left']
}

slice_indices = [4, 18, 20, 22]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [23]:
data = {'1-0': ['9C5093A5_left']
}

slice_indices = [5,13,18,23]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [24]:
data = {'0-0': ['ODELIA_BRAID1_0246_1_left'],
}

slice_indices = [8, 15, 16, 17]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [14]:
data = {'2-2': ['2983978521_left'],
}

slice_indices = [18,19,20,21]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [15]:
data = {'2-1': ['UKA_64_right'],
}

slice_indices = [18,20,22,23]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [17]:
data = {'1-1': ['3E6C31E4_left'],
}

slice_indices = [18,19,20,21]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [18]:
data = {'1-2': ['EF11BBDA_right'],
}

slice_indices = [19,20,26,27]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [19]:
data = {'0-1': ['3E6C31E4_right'],
}

slice_indices = [10,14,24,27]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.


In [21]:
data = {'0-2': ['RUMC_035_right'],
}

slice_indices = [9,12,17,19]  # choose manually

save_selected_slice_comparison(
    data=data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image",
    slice_indices=slice_indices
)

Finished.
